# KMeans — Regionais de Saúde de MG (painel Regional x Ano)

Unidade de agrupamento: par **(REGIONAL DE SAÚDE, ANO)**. Uma mesma Regional pode cair em agrupamentos diferentes em anos diferentes -- isso é informação (mudança de perfil epidemiológico ao longo do tempo), não inconsistência.

Mesmas regras de decisão do notebook de DBSCAN equivalente (`DBSCAN/dbscan_regionaisManual.ipynb`) -- montagem de painel a partir dos CSVs de taxa, janela comum, nenhum nulo imputado, RobustScaler + PCA condicional -- mas replicadas de forma INDEPENDENTE: este notebook não importa nada da pasta `DBSCAN/`, para que os dois algoritmos possam ser configurados e editados sem um afetar o outro. O que muda entre os dois é só o algoritmo de agrupamento: em vez de densidade (DBSCAN), o KMeans particiona o espaço em **k** regiões ao redor de k centróides, minimizando a inércia (soma dos quadrados das distâncias ao centróide). Isso troca a lógica de eps/MinPts por uma única escolha: **k**, feita pelo cotovelo da curva de inércia (Thorndike, 1953) -- também não fixado à mão.

Toda a lógica reutilizável está em `utilitarios_agrupamento_kmeansManual.py` (pipeline) e `graficos_agrupamento_kmeansManual.py` (visualizações), ambos nesta pasta; este notebook só configura e executa. Nenhum resultado é gravado em disco -- matriz, atributos e agrupamento ficam só como saída de célula, como no DBSCAN.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd()))


from utilitarios_agrupamento_kmeansManual import (
    COLUNA_ANO, COLUNA_REGIONAL,
    BLOCOS_DE_TAXA, PASTA_TAXA,
    LIMIAR_MAXIMO_DE_NULOS_POR_COLUNA,
    USAR_PCA, RAZAO_MAXIMA_ATRIBUTOS_POR_LINHA, VARIANCIA_EXPLICADA_ALVO,
    NUMERO_MAXIMO_DE_COMPONENTES, SEMENTE_ALEATORIA,
    USAR_JANELA_COMUM, ANO_INICIAL_COMUM, ANO_FINAL_COMUM,
    K_MINIMO, K_MAXIMO, SEMENTES_DA_BUSCA, MINIMO_DE_ELEMENTOS_POR_CLUSTER,
    montar_painel, calcular_janela_comum, restringir_a_janela_comum,
    descartar_colunas_sem_informacao_e_total, descartar_colunas_com_muitos_nulos, tratar_nulos_sem_imputar,
    padronizar_e_reduzir, preparar_projecao_2d, pesos_das_variaveis_no_componente,
    calcular_inercia_e_silhueta_por_k, escolher_k_pelo_cotovelo, busca_em_grade_k,
    executar_kmeans, resumir_agrupamento_kmeans, diagnostico_tamanho_dos_clusters,
    montar_resumo_parametros_kmeans, montar_tabela_pertencimento, montar_visao_agregada_regional_ano,
    montar_resumo_por_cluster,
    registrar_log,
)
from graficos_agrupamento_kmeansManual import (
    grafico_variancia_explicada, grafico_pesos_da_componente,
    grafico_cotovelo_inercia, grafico_silhueta_por_k, grafico_dispersao_pc1_pc2_kmeans,
    grafico_dispersao_pc1_pc2_por_ano,
    grafico_distribuicao_por_regional, grafico_distribuicao_por_ano,
    grafico_heatmap_regional_ano,
)

## 1. Leitura dos CSVs de taxa e reformatação (largo -> painel)

In [2]:
painel = montar_painel(blocos=BLOCOS_DE_TAXA, pasta_taxa=PASTA_TAXA)
painel.head()

[LOG] tabela_casos_por_faixa_etária_taxa.csv: 9 categoria(s), 1064 par(es) regional-ano.
[LOG] tabela_casos_por_escolaridade_taxa.csv: 8 categoria(s), 728 par(es) regional-ano.
[LOG] tabela_casos_por_raça_e_cor_taxa.csv: 7 categoria(s), 728 par(es) regional-ano.
[LOG] tabela_mortes_por_faixa_etária_taxa.csv: 9 categoria(s), 1288 par(es) regional-ano.
[LOG] tabela_mamografias_por_faixa_etária_taxa.csv: 9 categoria(s), 364 par(es) regional-ano.
[LOG] Painel montado: 1316 par(es) regional-ano x 42 variável(is), 28 regional(is).


casos-faixa_0-9  casos-faixa_10-19  casos-faixa_20-29  \
REGIONAL DE SAÚDE ANO                                                           
alfenas           1979              NaN                NaN                NaN   
                  1980              NaN                NaN                NaN   
                  1981              NaN                NaN                NaN   
                  1982              NaN                NaN                NaN   
                  1983              NaN                NaN                NaN   

                        casos-faixa_30-39  casos-faixa_40-49  \
REGIONAL DE SAÚDE ANO                                          
alfenas           1979                NaN                NaN   
                  1980                NaN                NaN   
                  1981                NaN                NaN   
                  1982                NaN                NaN   
                  1983                NaN                NaN   

                        casos-faixa_50-59  casos-faixa_60-69  \
REGIONAL DE SAÚDE ANO                                          
alfenas           1979                NaN                NaN   
                  1980                NaN                NaN   
                  1981                NaN                NaN   
                  1982                NaN                NaN   
                  1983                NaN                NaN   

                        casos-faixa_70-ou-mais  casos-faixa_total  \
REGIONAL DE SAÚDE ANO                                               
alfenas           1979                     NaN                NaN   
                  1980                     NaN                NaN   
                  1981                     NaN                NaN   
                  1982                     NaN                NaN   
                  1983                     NaN                NaN   

                        casos-escolaridade_fundamental-completo  ...  \
REGIONAL DE SAÚDE ANO                                            ...   
alfenas           1979                                      NaN  ...   
                  1980                                      NaN  ...   
                  1981                                      NaN  ...   
                  1982                                      NaN  ...   
                  1983                                      NaN  ...   

                        mortalidade_total  mamografia_0-9  mamografia_10-19  \
REGIONAL DE SAÚDE ANO                                                         
alfenas           1979               3.54             NaN               NaN   
                  1980               7.47             NaN               NaN   
                  1981               3.99             NaN               NaN   
                  1982               3.38             NaN               NaN   
                  1983               7.82             NaN               NaN   

                        mamografia_20-29  mamografia_30-39  mamografia_40-49  \
REGIONAL DE SAÚDE ANO                                                          
alfenas           1979               NaN               NaN               NaN   
                  1980               NaN               NaN               NaN   
                  1981               NaN               NaN               NaN   
                  1982               NaN               NaN               NaN   
                  1983               NaN               NaN               NaN   

                        mamografia_50-59  mamografia_60-69  \
REGIONAL DE SAÚDE ANO                                        
alfenas           1979               NaN               NaN   
                  1980               NaN               NaN   
                  1981               NaN               NaN   
                  1982               NaN               NaN   
                  1983               NaN               NaN   

                   

## 2. Janela temporal, descarte de colunas e tratamento de nulos

`USAR_JANELA_COMUM=True` restringe o painel à interseção dos períodos declarados de cada bloco (ou ao período manual `ANO_INICIAL_COMUM`/`ANO_FINAL_COMUM`, se informado); `False` usa o período completo de cada bloco, sem cortar ano nenhum.

Nos CSVs de taxa, uma célula fica em branco (NaN) quando a soma bruta da categoria-ano-regional é NaN em todos os municípios -- `calculo_taxa.ipynb` preserva essa ambiguidade de propósito (`sum(min_count=1)`): não dá para distinguir "zero eventos" de "sem população válida para calcular" só olhando a taxa. Por isso **nenhum nulo vira 0 aqui** -- mesma regra do DBSCAN, para os dois algoritmos serem comparáveis sobre a mesma matriz. A sequência abaixo só remove o que é estrutural (fora da janela, categoria sem-informação/total, atributo quase todo nulo) e descarta as linhas que ainda restarem com algum valor ausente -- sem preencher nada.

In [3]:
if USAR_JANELA_COMUM:
    janela = calcular_janela_comum(BLOCOS_DE_TAXA, ano_inicial_manual=ANO_INICIAL_COMUM, ano_final_manual=ANO_FINAL_COMUM)
    matriz = restringir_a_janela_comum(painel, janela)
else:
    registrar_log("USAR_JANELA_COMUM=False: usando o período completo de cada bloco, sem restringir ano.")
    matriz = painel

matriz = descartar_colunas_sem_informacao_e_total(matriz)
matriz = descartar_colunas_com_muitos_nulos(matriz, limiar_maximo=LIMIAR_MAXIMO_DE_NULOS_POR_COLUNA)
matriz = tratar_nulos_sem_imputar(matriz)

registrar_log(f"Matriz final: {matriz.shape[0]} linha(s) x {matriz.shape[1]} atributo(s).")
matriz.head()

[LOG] Janela comum: 2013-2023 (definida manualmente).
[LOG] Restrita à janela comum 2013-2023: 308 de 1316 par(es) regional-ano.
[LOG] Descartada(s) 7 coluna(s) sem-informação/total: ['casos-faixa_total', 'casos-escolaridade_sem-informacao', 'casos-escolaridade_total', 'casos-raca-cor_sem-informacao', 'casos-raca-cor_total', 'mortalidade_total', 'mamografia_total']
[LOG] Descartada(s) 3 coluna(s) com mais de 70% de nulos: ['mortalidade_0-9', 'mortalidade_10-19', 'mortalidade_20-29']
[LOG] Descartada(s) 194 de 308 linha(s) (63.0%) por ter ao menos um valor ausente -- nenhum valor foi imputado.
[LOG] Matriz final: 114 linha(s) x 32 atributo(s).


casos-faixa_0-9  casos-faixa_10-19  casos-faixa_20-29  \
REGIONAL DE SAÚDE ANO                                                           
alfenas           2015              0.0                0.0              11.37   
                  2018              0.0                0.0               0.00   
                  2020              0.0                0.0               0.00   
                  2021              0.0                0.0              15.04   
                  2023              0.0                0.0               6.13   

                        casos-faixa_30-39  casos-faixa_40-49  \
REGIONAL DE SAÚDE ANO                                          
alfenas           2015              76.55             150.01   
                  2018              49.72             132.57   
                  2020              86.79              74.51   
                  2021              41.91             103.75   
                  2023              61.31             146.59   

                        casos-faixa_50-59  casos-faixa_60-69  \
REGIONAL DE SAÚDE ANO                                          
alfenas           2015             204.35             177.21   
                  2018             118.94             140.00   
                  2020             118.04             167.39   
                  2021             173.92             157.07   
                  2023             164.66             138.01   

                        casos-faixa_70-ou-mais  \
REGIONAL DE SAÚDE ANO                            
alfenas           2015                  251.93   
                  2018                  147.80   
                  2020                  171.10   
                  2021                  140.78   
                  2023                  204.25   

                        casos-escolaridade_fundamental-completo  \
REGIONAL DE SAÚDE ANO                                             
alfenas           2015                                    23.77   
                  2018                                    34.26   
                  2020                                    41.36   
                  2021                                    41.29   
                  2023                                    29.31   

                        casos-escolaridade_fundamental-incompleto  ...  \
REGIONAL DE SAÚDE ANO                                              ...   
alfenas           2015                                      40.22  ...   
                  2018                                      10.23  ...   
                  2020                                       6.53  ...   
                  2021                                      13.33  ...   
                  2023                                      28.05  ...   

                        mortalidade_60-69  mortalidade_70-ou-mais  \
REGIONAL DE SAÚDE ANO                                               
alfenas           2015              10.13                   50.39   
                  2018              18.06                   41.38   
                  2020              46.03                   67.40   
                  2021              24.17                  106.80   
                  2023              18.65                   59.57   

                        mamografia_0-9  mamografia_10-19  mamografia_20-29  \
REGIONAL DE SAÚDE ANO                                                        
alfenas           2015            0.00              0.00              0.00   
                  2018            0.00             12.74            125.76   
                  2020            7.48             16.73            160.93   
                  2021            7.56              3.43             75.20   
                  2023            0.00              7.20             39.84   

                        mamografia_30-39  mamografia_40-49  mamografia_50-59  \
REGIONAL DE SAÚDE ANO                                                          
alfenas           2015     

## 3. Padronização (RobustScaler) e PCA

`USAR_PCA=None` decide sozinho pela razão atributos/linhas (`RAZAO_MAXIMA_ATRIBUTOS_POR_LINHA`); `True` força PCA; `False` pula o PCA inteiramente -- em ambos os casos o critério automático é ignorado. Mesma configuração do DBSCAN.

In [4]:
matriz_padronizada, matriz_para_kmeans, pca, variancia_acumulada = padronizar_e_reduzir(
    matriz,
    razao_maxima_atributos_por_linha=RAZAO_MAXIMA_ATRIBUTOS_POR_LINHA,
    variancia_alvo=VARIANCIA_EXPLICADA_ALVO,
    numero_maximo_de_componentes=NUMERO_MAXIMO_DE_COMPONENTES,
    semente=SEMENTE_ALEATORIA,
    usar_pca=USAR_PCA,
)

[LOG] PCA não aplicado: USAR_PCA=False (forçado; razão atributos/linhas = 0.281).


### Diagnóstico: variância por componente e pesos das variáveis

Só roda quando o PCA foi de fato aplicado (`pca is not None`). O "scree plot" mostra a variância explicada por componente, com PC1 destacado em vermelho se ultrapassar `VARIANCIA_PC1_ALERTA` -- direção dominante isolada sugere que o componente captura porte/volume da regional, não perfil epidemiológico. Os pesos ("loadings") de PC1 mostram quais atributos originais mais influenciam esse eixo.

In [5]:
if pca is not None:
    grafico_variancia_explicada(pca, variancia_acumulada).show()
    display(pesos_das_variaveis_no_componente(pca, matriz.columns, componente=1))
    grafico_pesos_da_componente(pca, matriz.columns, componente=1).show()
else:
    registrar_log("USAR_PCA resultou em pca=None -- diagnóstico de variância/pesos não se aplica (KMeans rodou sobre os atributos originais).")

[LOG] USAR_PCA resultou em pca=None -- diagnóstico de variância/pesos não se aplica (KMeans rodou sobre os atributos originais).


## 4. Escolha de k (cotovelo da curva de inércia)

In [6]:
tabela_k = calcular_inercia_e_silhueta_por_k(matriz_para_kmeans, k_minimo=K_MINIMO, k_maximo=K_MAXIMO, semente=SEMENTE_ALEATORIA)
k_escolhido, indice_do_cotovelo = escolher_k_pelo_cotovelo(tabela_k)

figura_cotovelo = grafico_cotovelo_inercia(tabela_k, indice_do_cotovelo, k_escolhido)
figura_cotovelo.show()
tabela_k

[LOG] Inércia e silhueta calculadas para k = 2..10 (9 valor(es) de k).
[LOG] Cotovelo da curva de inércia na posição 3 de 9 -- k = 5


,k,inercia,silhueta
0,2,2776.682356,0.636431
1,3,2376.465051,0.194890
2,4,2147.007058,0.178891
3,5,1954.549495,0.133764
4,6,1839.655467,0.090315
5,7,1770.670657,0.092789
6,8,1682.507744,0.100562
7,9,1583.587979,0.104729
8,10,1533.835103,0.099710


### Silhueta por k (validação complementar)

In [7]:
figura_silhueta = grafico_silhueta_por_k(tabela_k, k_escolhido)
figura_silhueta.show()

### Busca em grade (validação complementar)

In [8]:
tabela_da_busca = busca_em_grade_k(matriz_para_kmeans, k_minimo=K_MINIMO, k_maximo=K_MAXIMO, sementes=SEMENTES_DA_BUSCA)
tabela_da_busca.head(20)

,k,semente,inercia,silhueta
0,2,42,2776.682356,0.636431
1,3,42,2376.465051,0.194890
2,4,42,2147.007058,0.178891
3,5,42,1954.549495,0.133764
4,6,42,1839.655467,0.090315
5,7,42,1770.670657,0.092789
6,8,42,1682.507744,0.100562
7,9,42,1583.587979,0.104729
8,10,42,1533.835103,0.099710


## 5. KMeans

In [9]:
rotulos, modelo_kmeans = executar_kmeans(matriz_para_kmeans, k_escolhido, semente=SEMENTE_ALEATORIA)
n_clusters, inercia_final, silhueta_final = resumir_agrupamento_kmeans(matriz_para_kmeans, rotulos, modelo_kmeans)

registrar_log(f"KMeans(k={k_escolhido}, random_state={SEMENTE_ALEATORIA})")
registrar_log(f"  Agrupamentos: {n_clusters}")
registrar_log(f"  Inércia (WCSS): {inercia_final:.4f}")
registrar_log(f"  Silhueta: {silhueta_final:.4f}" if not np.isnan(silhueta_final) else "  Silhueta: não aplicável (k < 2)")

_ = diagnostico_tamanho_dos_clusters(rotulos, minimo=MINIMO_DE_ELEMENTOS_POR_CLUSTER)

[LOG] KMeans(k=5, random_state=42)
[LOG]   Agrupamentos: 5
[LOG]   Inércia (WCSS): 1954.5495
[LOG]   Silhueta: 0.1338
[LOG] ATENÇÃO: 2 cluster(s) com menos de 4 elemento(s) -- {3: 1, 4: 2}. O KMeans aloca todo ponto a algum cluster (não existe 'ruído' como no DBSCAN); um cluster muito pequeno tende a ser um outlier isolado, não um perfil compartilhado -- considere reduzir k ou tratar esses pontos como atípicos na interpretação.


## Visualizações

In [10]:
matriz_2d, variancia_explicada_2d = preparar_projecao_2d(matriz_padronizada, matriz_para_kmeans, pca, semente=SEMENTE_ALEATORIA)

figura_dispersao = grafico_dispersao_pc1_pc2_kmeans(matriz_2d, rotulos, matriz.index, variancia_explicada_2d, k_escolhido)
figura_dispersao.show()

### Dispersão PC1 x PC2 por ano

Mesmos pontos e paleta do gráfico acima, mas em uma grade de mini-gráficos -- um por ano -- para comparar a composição dos agrupamentos ano a ano em vez de ver todos os pontos misturados.

In [11]:
figura_dispersao_por_ano = grafico_dispersao_pc1_pc2_por_ano(matriz_2d, rotulos, matriz.index, variancia_explicada_2d)
figura_dispersao_por_ano.show()

In [12]:
resultado = montar_tabela_pertencimento(matriz.index, rotulos)

figura_regional = grafico_distribuicao_por_regional(resultado)
figura_regional.show()

figura_ano = grafico_distribuicao_por_ano(resultado)
figura_ano.show()

### Composição de cada cluster (tabela, não só o gráfico)

A dispersão acima projeta em PC1 x PC2 -- dois clusters que o KMeans separou no espaço original podem aparecer sobrepostos no 2D. Esta tabela lista, por CLUSTER, quantos pares e quais Regional-Ano pertencem a ele, direto dos rótulos do KMeans, sem depender da projeção. Diferente do DBSCAN, não há rótulo de ruído -- todo par pertence a algum cluster.

In [13]:
resumo_por_cluster = montar_resumo_por_cluster(resultado)
resumo_por_cluster

,CLUSTER,N_PARES,N_REGIONAIS_DISTINTAS,MEMBROS
0,0,55,24,"belo horizonte (2015), belo horizonte (2020), ..."
1,1,26,13,"alfenas (2015), alfenas (2021), alfenas (2023)..."
2,2,30,17,"alfenas (2018), alfenas (2020), barbacena (201..."
3,3,1,1,leopoldina (2020)
4,4,2,1,"passos (2021), passos (2023)"


## Saídas finais

In [14]:
resumo_parametros = montar_resumo_parametros_kmeans(
    k_escolhido, inercia_final, silhueta_final,
    usar_robust_scaler=True, pca=pca, variancia_acumulada=variancia_acumulada,
    n_linhas=matriz.shape[0], n_atributos=matriz.shape[1], n_clusters=n_clusters,
)
resumo_parametros

,K,ESCALONADOR,PCA_APLICADO,PCA_N_COMPONENTES,PCA_VARIANCIA_EXPLICADA,N_LINHAS,N_ATRIBUTOS,N_CLUSTERS,INERCIA,SILHUETA
0,5,RobustScaler,False,NaN,NaN,114,32,5,1954.549495,0.133764


In [15]:
atributos_usados = pd.DataFrame({"ATRIBUTO": list(matriz.columns)})
atributos_usados

,ATRIBUTO
0,casos-faixa_0-9
1,casos-faixa_10-19
2,casos-faixa_20-29
3,casos-faixa_30-39
4,casos-faixa_40-49
5,casos-faixa_50-59
6,casos-faixa_60-69
7,casos-faixa_70-ou-mais
8,casos-escolaridade_fundamental-completo
9,casos-escolaridade_fundamental-incompleto


In [16]:
resultado

,REGIONAL DE SAÚDE,ANO,CLUSTER
0,alfenas,2015,1
1,alfenas,2018,2
2,alfenas,2020,2
3,alfenas,2021,1
4,alfenas,2023,1
...,...,...,...
109,varginha,2015,2
110,varginha,2018,1
111,varginha,2020,0
112,varginha,2021,0


In [17]:
visao_agregada_regional_ano = montar_visao_agregada_regional_ano(resultado)
visao_agregada_regional_ano

ANO,2015,2018,2020,2021,2023
REGIONAL DE SAÚDE,,,,,
alfenas,1.0,2.0,2.0,1.0,1.0
barbacena,NaN,2.0,2.0,NaN,NaN
belo horizonte,0.0,2.0,0.0,2.0,0.0
coronel fabriciano,0.0,1.0,0.0,0.0,2.0
diamantina,0.0,2.0,NaN,0.0,2.0
divinopolis,0.0,0.0,0.0,0.0,0.0
governador valadares,0.0,0.0,0.0,0.0,1.0
itabira,0.0,2.0,0.0,0.0,1.0
ituiutaba,0.0,NaN,0.0,0.0,NaN


### Heatmap: cluster por Regional e Ano

Mesma informação de `visao_agregada_regional_ano`, mas como cor em vez de número -- uma cor por CLUSTER, célula em branco onde não há dado (par descartado no dropna ou fora da cobertura de algum bloco).

In [18]:
figura_heatmap = grafico_heatmap_regional_ano(visao_agregada_regional_ano)
figura_heatmap.show()